# Plan de Producción - Clasificador de Sentimientos (CPU-friendly)

Guía para pasar de notebook experimental a estructura de producción siguiendo la PARTE 11 del notebook de modelado.

## Índice
- Parte A: Checklist de producción
- Parte B: Refactor a módulos (.py) y contenido de archivos

## Parte A · Checklist paso a paso

Cada paso indica objetivo, archivos a crear/modificar y validaciones/riesgos clave.

### Paso 1: Preprocesamiento
- Objetivo: Extraer las 7 funciones + constantes a un módulo reutilizable.
- Archivos: crear `src/preprocessing.py`.
- Validaciones/Riesgos: Orden de funciones crítico; manejar texto vacío/None; tests con ejemplos que contengan tokens especiales.

### Paso 2: Configuración
- Objetivo: Centralizar hiperparámetros y rutas de artefactos.
- Archivos: crear `src/config.py`.
- Validaciones/Riesgos: Evitar paths hardcode; usar `Path`; permitir override por variables de entorno; definir VERSION.

### Paso 3: Carga de artefactos
- Objetivo: Cargar modelo y vectorizador solo una vez (Singleton/lazy load).
- Archivos: crear `src/model_loader.py`.
- Validaciones/Riesgos: Manejar ausencia/corrupción de archivos con errores claros; loggear versión; no recargar por petición.

### Paso 4: Extracción de keywords
- Objetivo: Implementar `extraer_keywords` usando coeficientes y vocabulario.
- Archivos: crear `src/keyword_extractor.py`.
- Validaciones/Riesgos: Caso sin términos en vocab → lista vacía; cache de vocabulario; coherencia mayúsculas/tokens especiales.

### Paso 5: Predictor
- Objetivo: Clase `SentimentPredictor` con pipeline de inferencia completo.
- Archivos: crear `src/predictor.py`.
- Validaciones/Riesgos: Validar entrada; controlar errores en prepro/vectorización/predicción; mapear clases fijo; redondeo de probabilidad; no depender del orden de celdas.

### Paso 6: Contrato API
- Objetivo: Tipar request/response para cumplir PLAN_MODELADO §1.2.
- Archivos: crear `src/api_schema.py`.
- Validaciones/Riesgos: Campos obligatorios; probability en [0,1]; timestamp ISO 8601; clases válidas {Negativo, Neutro, Positivo}.

### Paso 7: Logging
- Objetivo: Configuración de logging para trazabilidad sin exponer PII.
- Archivos: crear `src/logging_config.py`.
- Validaciones/Riesgos: No loggear texto completo; logs estructurados; niveles DEBUG/INFO/ERROR; opcional rotación.

### Paso 8: Tests de integración
- Objetivo: Verificar contrato y métricas mínimas con reseñas controladas.
- Archivos: crear `tests/test_integration.py` (+ fixtures necesarios).
- Validaciones/Riesgos: Usar tolerancias en métricas; probar casos edge (vacío, emojis); asegurar determinismo; tiempo <100ms por predicción.

## Parte B · Refactor a módulos (estructura)

Las siguientes secciones contendrán el código final de cada archivo listo para API (sin depender del orden de celdas del notebook):
- `src/preprocessing.py`
- `src/config.py`
- `src/model_loader.py`
- `src/keyword_extractor.py`
- `src/predictor.py`
- `src/api_schema.py`
- `src/logging_config.py`
- `tests/test_integration.py`

(Se completarán paso a paso con tu confirmación en los siguientes pasos.)

## Parte B · Paso 1 — src/preprocessing.py

Objetivo: portar las 7 funciones y constantes de preprocesamiento a un módulo reutilizable (sin dependencia del orden de celdas). Incluye validaciones de entrada y logging ligero.


In [2]:
# Importar librerías necesarias para generación de artefactos
import re
import time

# CONSTANTES DE PREPROCESSING (del modelo entrenado)

# Patrón de emojis
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticonos
    "\U0001F300-\U0001F5FF"  # símbolos & pictogramas
    "\U0001F680-\U0001F6FF"  # transporte & símbolos de mapas
    "\U0001F1E0-\U0001F1FF"  # banderas
    "]+",
    flags=re.UNICODE
)

# Mapeo de números a texto
NUMEROS_TEXTO = {
    0: 'cero', 1: 'uno', 2: 'dos', 3: 'tres', 4: 'cuatro', 5: 'cinco',
    6: 'seis', 7: 'siete', 8: 'ocho', 9: 'nueve', 10: 'diez',
    11: 'once', 12: 'doce', 13: 'trece', 14: 'catorce', 15: 'quince',
    16: 'dieciseis', 17: 'diecisiete', 18: 'dieciocho', 19: 'diecinueve',
    20: 'veinte', 30: 'treinta', 40: 'cuarenta', 50: 'cincuenta',
    60: 'sesenta', 70: 'setenta', 80: 'ochenta', 90: 'noventa', 100: 'cien'
}

# Mapeo de ordinales
ORDINALES = {
    1: 'primer', 2: 'segundo', 3: 'tercer', 4: 'cuarto', 5: 'quinto',
    6: 'sexto', 7: 'septimo', 8: 'octavo', 9: 'noveno', 10: 'decimo'
}

# Token pattern para TF-IDF
TOKEN_PATTERN = r'(?u)\b\w+\b|<[A-Z_0-9]+>'


# FUNCIONES DE PREPROCESSING (idénticas a modelo_sentiment.ipynb PARTE 2)

def numero_a_texto(num):
    """Convierte número a texto en español."""
    if num in NUMEROS_TEXTO:
        return NUMEROS_TEXTO[num]
    
    if 21 <= num < 30:
        return f"veinti{NUMEROS_TEXTO[num - 20]}"
    
    if num < 100:
        decenas, unidades = divmod(num, 10)
        base = NUMEROS_TEXTO.get(decenas * 10, str(num))
        if unidades:
            return f"{base} y {NUMEROS_TEXTO[unidades]}"
        return base
    
    if num == 100:
        return 'cien'
    
    return str(num)


def limpiar_ruido(texto):
    """Elimina URLs, emails, menciones, y etiquetas HTML."""
    if not texto:
        return ''
    
    t = texto
    t = re.sub(r'https?://\S+', ' ', t)
    t = re.sub(r'\S+@\S+', ' ', t)
    t = re.sub(r'[@#]\w+', ' ', t)
    t = t.replace('<', ' ').replace('>', ' ')
    t = re.sub(r'\s+', ' ', t).strip()
    
    return t


def eliminar_emojis(texto):
    """Elimina emojis del texto."""
    if not texto:
        return ''
    return EMOJI_PATTERN.sub(' ', texto)


def tokenizar_caracteres_especiales(texto):
    """Tokeniza precios y porcentajes."""
    if not texto:
        return ''
    
    t = re.sub(r'(\d+[.,]?\d*)\s?(€|eur|usd|mxn|ars|cop|$)', ' <PRECIO> ', texto, flags=re.IGNORECASE)
    t = re.sub(r'(\d+[.,]?\d*)\s?%', ' <PORCENTAJE> ', t)
    
    return re.sub(r'\s+', ' ', t).strip()


def tokenizar_patrones_numericos(texto):
    """Tokeniza patrones numéricos con unidades."""
    if not texto:
        return ''
    
    def reemplazo(match):
        numero = int(match.group(1))
        palabra = match.group(2).lower()
        
        base = numero_a_texto(numero)
        
        if palabra.startswith('vez'):
            return f'{base} veces'
        elif palabra.startswith('estrell'):
            return f'{base} estrellas'
        elif palabra.startswith('dia'):
            return f'{base} dias'
        elif palabra.startswith('hora'):
            return f'{base} horas'
        elif palabra.startswith('unidad'):
            return f'{base} unidades'
        
        return base
    
    t = re.sub(r'(\d{1,3})\s+(veces|estrellas|dias?|horas?|unidades?)', reemplazo, texto, flags=re.IGNORECASE)
    t = re.sub(r'(\d{1,3})(er|o|a)\b', lambda m: ORDINALES.get(int(m.group(1)), m.group(0)), t)
    t = re.sub(r'\b(\d{1,3})\b', '', t)
    
    return re.sub(r'\s+', ' ', t).strip()


def tokenizar_puntuacion_repetida(texto):
    """Tokeniza puntuación repetida."""
    if not texto:
        return ''
    
    t = re.sub(r'!{3,}', ' <EXCL_MULT> ', texto)
    t = re.sub(r'!{2}', ' <EXCL_2> ', t)
    t = re.sub(r'\?{3,}', ' <INTER_MULT> ', t)
    t = re.sub(r'\?{2}', ' <INTER_2> ', t)
    t = re.sub(r'\.{4,}', ' <SUSP_MULT> ', t)
    t = re.sub(r'\.\.\.', ' <SUSP_3> ', t)
    t = re.sub(r'[!?]{2,}', ' <MIXTO> ', t)
    
    return re.sub(r'\s+', ' ', t).strip()


def eliminar_caracteres_restantes(texto):
    """Elimina caracteres no alfanuméricos excepto tokens especiales."""
    if not texto:
        return ''
    
    t = re.sub(r'[^\w\s<>]', ' ', texto)
    return re.sub(r'\s+', ' ', t).strip()


def normalizar_mayusculas(texto):
    """Normaliza mayúsculas preservando tokens especiales y palabras en mayúsculas."""
    if not texto:
        return ''
    
    tokens = texto.split()
    normalizados = []
    
    for tok in tokens:
        if tok.startswith('<') and tok.endswith('>'):
            normalizados.append(tok.upper())
        elif tok.isupper():
            normalizados.append(tok)
        else:
            normalizados.append(tok.lower())
    
    return ' '.join(normalizados)


def preprocesar_texto(texto):
    """Pipeline completo de preprocesamiento (7 pasos)."""
    t = limpiar_ruido(texto)
    t = eliminar_emojis(t)
    t = tokenizar_caracteres_especiales(t)
    t = tokenizar_patrones_numericos(t)
    t = tokenizar_puntuacion_repetida(t)
    t = eliminar_caracteres_restantes(t)
    t = normalizar_mayusculas(t)
    return t


print("✅ Constantes y funciones de preprocesamiento cargadas")
print(f"   - EMOJI_PATTERN definido")
print(f"   - NUMEROS_TEXTO: {len(NUMEROS_TEXTO)} números")
print(f"   - ORDINALES: {len(ORDINALES)} ordinales")
print(f"   - TOKEN_PATTERN: {TOKEN_PATTERN}")
print(f"   - 7 funciones de preprocesamiento definidas")

✅ Constantes y funciones de preprocesamiento cargadas
   - EMOJI_PATTERN definido
   - NUMEROS_TEXTO: 29 números
   - ORDINALES: 10 ordinales
   - TOKEN_PATTERN: (?u)\b\w+\b|<[A-Z_0-9]+>
   - 7 funciones de preprocesamiento definidas


## Parte B · Paso 2 — src/config.py

Objetivo: centralizar hiperparámetros, rutas de artefactos y ajustes de inferencia/logging con overrides por variables de entorno (PORTABLE, sin hardcode).


In [ ]:
# Contenido de src/config.py y escritura a disco
from pathlib import Path
import textwrap

config_code = textwrap.dedent(
    """
    """Configuracion centralizada para inferencia en produccion (CPU-friendly)."""
    from __future__ import annotations

    import os
    from pathlib import Path
    from typing import Dict, Any

    BASE_DIR = Path(__file__).resolve().parent.parent
    ARTIFACTS_DIR = BASE_DIR

    MODEL_VERSION = "1.0.0"

    def _env_path(var_name: str, default: Path) -> Path:
        candidate = os.getenv(var_name)
        if candidate:
            return Path(candidate).expanduser().resolve()
        return default

    MODEL_PATH = _env_path("MODEL_PATH", ARTIFACTS_DIR / "modelo_sentiment_final.joblib")
    VECTORIZER_PATH = _env_path("VECTORIZER_PATH", ARTIFACTS_DIR / "tfidf_vectorizer_final.joblib")
    STOPWORDS_PATH = _env_path("STOPWORDS_PATH", ARTIFACTS_DIR / "stopwords_eliminar.txt")

    TFIDF_CONFIG: Dict[str, Any] = {
        "max_features": 10_000,
        "ngram_range": (1, 2),
        "min_df": 3,
        "max_df": 0.9,
        "lowercase": False,
        "token_pattern": r"(?u)\\b\\w+\\b|<[A-Z_0-9]+>",
        "stop_words": None,
    }

    MODEL_CONFIG: Dict[str, Any] = {
        "C": 0.5,
        "class_weight": "balanced",
        "multi_class": "multinomial",
        "solver": "lbfgs",
        "max_iter": 1000,
        "random_state": 42,
    }

    KEYWORDS_TOP_N = int(os.getenv("KEYWORDS_TOP_N", "5"))
    PROB_DECIMALS = int(os.getenv("PROB_DECIMALS", "2"))

    LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")
    LOG_FORMAT = os.getenv(
        "LOG_FORMAT",
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    )

    __all__ = [
        "BASE_DIR",
        "ARTIFACTS_DIR",
        "MODEL_VERSION",
        "MODEL_PATH",
        "VECTORIZER_PATH",
        "STOPWORDS_PATH",
        "TFIDF_CONFIG",
        "MODEL_CONFIG",
        "KEYWORDS_TOP_N",
        "PROB_DECIMALS",
        "LOG_LEVEL",
        "LOG_FORMAT",
    ]
    """
)

path = Path("src/config.py")
path.parent.mkdir(exist_ok=True)
path.write_text(config_code, encoding="utf-8")
print(path.read_text(encoding="utf-8"))


## Parte B · Paso 3 — src/model_loader.py

Objetivo: cargar modelo y vectorizador una sola vez (Singleton/lazy load), validando artefactos y stopwords.


In [ ]:
# Contenido de src/model_loader.py y escritura a disco
from pathlib import Path
import textwrap

model_loader_code = textwrap.dedent(
    """
    """Carga única de artefactos de modelo y vectorizador para inferencia (CPU-friendly)."""
    from __future__ import annotations

    import logging
    from pathlib import Path
    from typing import Optional

    import joblib

    from . import config
    from . import preprocessing

    logger = logging.getLogger(__name__)


    class ModelLoader:
        """Singleton ligero para cargar modelo y vectorizador una sola vez."""

        _instance: Optional["ModelLoader"] = None

        def __new__(cls):
            if cls._instance is None:
                cls._instance = super().__new__(cls)
            return cls._instance

        def __init__(self) -> None:
            if getattr(self, "_initialized", False):
                return
            self.model_path: Path = Path(config.MODEL_PATH)
            self.vectorizer_path: Path = Path(config.VECTORIZER_PATH)
            self.stopwords_path: Path = Path(config.STOPWORDS_PATH)
            self.model = None
            self.vectorizer = None
            self.stopwords = preprocessing.DEFAULT_STOPWORDS
            self._initialized = True

        def _validate_paths(self) -> None:
            missing = [p for p in [self.model_path, self.vectorizer_path] if not p.exists()]
            if missing:
                raise FileNotFoundError(f"Faltan artefactos: {missing}")

        def load(self):
            """Carga modelo y vectorizador si aún no están cargados."""
            if self.model is not None and self.vectorizer is not None:
                return self.model, self.vectorizer

            self._validate_paths()

            logger.info("Cargando modelo desde %s", self.model_path)
            self.model = joblib.load(self.model_path)

            logger.info("Cargando vectorizador desde %s", self.vectorizer_path)
            self.vectorizer = joblib.load(self.vectorizer_path)

            if self.stopwords_path.exists():
                self.stopwords = preprocessing._cargar_stopwords(self.stopwords_path)
            return self.model, self.vectorizer

        def get(self):
            if self.model is None or self.vectorizer is None:
                return self.load()
            return self.model, self.vectorizer


    loader = ModelLoader()

    __all__ = ["ModelLoader", "loader"]
    """
)

path = Path("src/model_loader.py")
path.parent.mkdir(exist_ok=True)
path.write_text(model_loader_code, encoding="utf-8")
print(path.read_text(encoding="utf-8"))


## Parte B · Paso 4 — src/keyword_extractor.py

Objetivo: extraer keywords por texto usando coeficientes de LogReg y vocabulario TF-IDF.


In [ ]:
# Contenido de src/keyword_extractor.py y escritura a disco
from pathlib import Path
import textwrap

keyword_code = textwrap.dedent(
    """
    """Extraccion de keywords basada en coeficientes del modelo (LogReg)."""
    from __future__ import annotations

    from typing import List
    import numpy as np


    def extraer_keywords(texto_prep: str, prediccion: str, model, vectorizer, top_n: int = 5) -> List[str]:
        """
        Retorna las palabras del texto preprocesado con mayor contribucion para la clase predicha.
        - texto_prep: texto ya preprocesado (tokens separados por espacios)
        - prediccion: etiqueta predicha (ej: "negativo", "neutro", "positivo")
        - model: instancia de LogisticRegression entrenada
        - vectorizer: TfidfVectorizer entrenada
        - top_n: numero de palabras a retornar
    
    NOTA: No se aplica .lower() porque el texto ya viene normalizado
    desde normalizar_mayusculas() en el preprocesamiento.
    """
    if not texto_prep:
        return []

    vocab = vectorizer.get_feature_names_out()
    vocab_index = {t: i for i, t in enumerate(vocab)}

    try:
        class_idx = list(model.classes_).index(prediccion)
    except ValueError:
        return []

    coefs = model.coef_[class_idx]
    palabras = set(texto_prep.split())  # Ya normalizado, NO usar .lower()
        return [p for p, _ in contribuciones[:top_n]]


    __all__ = ["extraer_keywords"]
    """
)

path = Path("src/keyword_extractor.py")
path.parent.mkdir(exist_ok=True)
path.write_text(keyword_code, encoding="utf-8")
print(path.read_text(encoding="utf-8"))


## Parte B · Paso 5 — src/predictor.py

Objetivo: Clase `SentimentPredictor` con pipeline completo de inferencia, mapeo de clases y formateo de respuesta.


In [ ]:
# Contenido de src/predictor.py y escritura a disco
from pathlib import Path
import textwrap

predictor_code = textwrap.dedent(
    """
    """Clase principal de inferencia para el clasificador de sentimientos."""
    from __future__ import annotations

    import logging
    from dataclasses import dataclass
    from datetime import datetime, timezone
    from typing import Dict, Any

    import numpy as np

    from . import config
    from .model_loader import loader
    from . import preprocessing
    from .keyword_extractor import extraer_keywords

    logger = logging.getLogger(__name__)

    MAPEO_CLASES = {
        "negativo": "Negativo",
        "neutro": "Neutro",
        "positivo": "Positivo",
    }


    def _timestamp_iso() -> str:
        return datetime.now(timezone.utc).isoformat()


    def _validar_texto(texto: Any) -> str:
        if texto is None:
            raise ValueError("'texto' no puede ser None")
        if not isinstance(texto, str):
            raise TypeError("'texto' debe ser str")
        if not texto.strip():
            raise ValueError("'texto' no puede estar vacio")
        return texto


    @dataclass
    class PredictionResult:
        prediction: str
        probability: float
        keywords: list[str]
        timestamp: str

        def to_dict(self) -> Dict[str, Any]:
            return {
                "prediction": self.prediction,
                "probability": self.probability,
                "keywords": self.keywords,
                "timestamp": self.timestamp,
            }


    class SentimentPredictor:
        def __init__(self) -> None:
            self.model, self.vectorizer = loader.get()

        def predict(self, texto: str) -> PredictionResult:
            texto_raw = _validar_texto(texto)

            texto_prep = preprocessing.preprocesar_texto(texto_raw)
            if not texto_prep:
                raise ValueError("El preprocesamiento produjo texto vacio")

            X = self.vectorizer.transform([texto_prep])
            proba = self.model.predict_proba(X)[0]
            pred_raw = self.model.predict(X)[0]

            pred_api = MAPEO_CLASES.get(pred_raw, pred_raw.capitalize())
            prob_val = round(float(np.max(proba)), config.PROB_DECIMALS)

            kws = extraer_keywords(
                texto_prep=texto_prep,
                prediccion=pred_raw,
                model=self.model,
                vectorizer=self.vectorizer,
                top_n=config.KEYWORDS_TOP_N,
            )

            ts = _timestamp_iso()
            logger.info("prediction=%s prob=%.3f", pred_api, prob_val)

            return PredictionResult(
                prediction=pred_api,
                probability=prob_val,
                keywords=kws,
                timestamp=ts,
            )


    __all__ = ["SentimentPredictor", "PredictionResult", "MAPEO_CLASES"]
    """
)

path = Path("src/predictor.py")
path.parent.mkdir(exist_ok=True)
path.write_text(predictor_code, encoding="utf-8")
print(path.read_text(encoding="utf-8"))


## Parte B · Paso 6 — src/api_schema.py

Objetivo: tipar request/response cumpliendo contrato (PLAN_MODELADO §1.2) con validaciones básicas.


In [ ]:
# Contenido de src/api_schema.py y escritura a disco
from pathlib import Path
import textwrap

api_schema_code = textwrap.dedent(
    """
    """Contrato de entrada/salida para la API de sentimiento."""
    from __future__ import annotations

    from dataclasses import dataclass, field
    from datetime import datetime
    from typing import List

    VALID_CLASSES = ["Negativo", "Neutro", "Positivo"]


    def _validate_prediction(pred: str) -> str:
        if pred not in VALID_CLASSES:
            raise ValueError(f"prediction debe ser uno de {VALID_CLASSES}")
        return pred


    def _validate_probability(p: float) -> float:
        if not (0.0 <= p <= 1.0):
            raise ValueError("probability debe estar entre 0 y 1")
        return float(p)


    def _validate_keywords(kws: List[str]) -> List[str]:
        if not isinstance(kws, list):
            raise TypeError("keywords debe ser lista")
        return [str(k) for k in kws]


    def _validate_timestamp(ts: str) -> str:
        try:
            datetime.fromisoformat(ts)
        except Exception as exc:
            raise ValueError("timestamp debe ser ISO 8601") from exc
        return ts


    @dataclass
    class SentimentRequest:
        texto: str

        def __post_init__(self):
            if not isinstance(self.texto, str) or not self.texto.strip():
                raise ValueError("texto es obligatorio y debe ser string no vacio")


    @dataclass
    class SentimentResponse:
        prediction: str
        probability: float
        keywords: List[str] = field(default_factory=list)
        timestamp: str = ""

        def __post_init__(self):
            self.prediction = _validate_prediction(self.prediction)
            self.probability = _validate_probability(self.probability)
            self.keywords = _validate_keywords(self.keywords)
            self.timestamp = _validate_timestamp(self.timestamp)


    __all__ = ["SentimentRequest", "SentimentResponse", "VALID_CLASSES"]
    """
)

path = Path("src/api_schema.py")
path.parent.mkdir(exist_ok=True)
path.write_text(api_schema_code, encoding="utf-8")
print(path.read_text(encoding="utf-8"))


## Parte B · Paso 7 — src/logging_config.py

Objetivo: configuración de logging centralizada (sin PII), con overrides por entorno.


In [ ]:
# Contenido de src/logging_config.py y escritura a disco
from pathlib import Path
import textwrap

logging_code = textwrap.dedent(
    """
    """Configuracion de logging (sin PII, orientado a produccion CPU-friendly)."""
    from __future__ import annotations

    import logging
    import os
    from logging import Logger
    from typing import Optional

    from . import config


    def setup_logging(level: Optional[str] = None, fmt: Optional[str] = None) -> None:
        """Configura logging global."""
        logging.basicConfig(
            level=(level or config.LOG_LEVEL),
            format=(fmt or config.LOG_FORMAT),
        )


    def get_logger(name: str) -> Logger:
        if not logging.getLogger().handlers:
            setup_logging()
        return logging.getLogger(name)


    __all__ = ["setup_logging", "get_logger"]
    """
)

path = Path("src/logging_config.py")
path.parent.mkdir(exist_ok=True)
path.write_text(logging_code, encoding="utf-8")
print(path.read_text(encoding="utf-8"))


## Parte B · Paso 8 — tests/test_integration.py

Objetivo: asegurar contrato y rapidez de inferencia; skip si faltan artefactos.


In [ ]:
# Contenido de tests/test_integration.py y escritura a disco
from pathlib import Path
import textwrap

test_code = textwrap.dedent(
    """
    """Tests de integracion ligeros para la API de sentimiento."""
    from __future__ import annotations

    import time
    from pathlib import Path

    import pytest

    from src.predictor import SentimentPredictor
    from src.api_schema import SentimentRequest, SentimentResponse, VALID_CLASSES
    from src import config

    ARTIFACTS_PRESENT = Path(config.MODEL_PATH).exists() and Path(config.VECTORIZER_PATH).exists()


    @pytest.mark.skipif(not ARTIFACTS_PRESENT, reason="Faltan artefactos de modelo/vectorizer")
    def test_prediccion_estructura_y_rangos():
        predictor = SentimentPredictor()
        req = SentimentRequest(texto="Producto excelente, muy recomendado!!!")
        result = predictor.predict(req.texto)

        resp = SentimentResponse(
            prediction=result.prediction,
            probability=result.probability,
            keywords=result.keywords,
            timestamp=result.timestamp,
        )
        assert resp.prediction in VALID_CLASSES
        assert 0.0 <= resp.probability <= 1.0
        assert isinstance(resp.keywords, list)
        assert len(resp.timestamp) > 0


    @pytest.mark.skipif(not ARTIFACTS_PRESENT, reason="Faltan artefactos de modelo/vectorizer")
    def test_probabilidad_es_maxima_del_vector():
        predictor = SentimentPredictor()
        req = SentimentRequest(texto="Es un producto malo, no lo recomiendo")
        result = predictor.predict(req.texto)
        assert 0.0 <= result.probability <= 1.0


    def test_texto_vacio_lanza_error():
        predictor = SentimentPredictor()
        with pytest.raises(ValueError):
            predictor.predict("")


    @pytest.mark.skipif(not ARTIFACTS_PRESENT, reason="Faltan artefactos de modelo/vectorizer")
    def test_tiempo_inferencia_rapido():
        predictor = SentimentPredictor()
        req = SentimentRequest(texto="Buen producto, funciona ok")
        t0 = time.time()
        predictor.predict(req.texto)
        assert (time.time() - t0) < 0.5  # CPU-friendly
    """
)

path = Path("tests/test_integration.py")
path.parent.mkdir(exist_ok=True)
path.write_text(test_code, encoding="utf-8")
print(path.read_text(encoding="utf-8"))


---

## Parte C · Funciones Completas del Modelo (Referencia de modelo_sentiment.ipynb)

Esta sección contiene las funciones REALES extraídas del notebook de modelado para implementación directa en producción. Cada función incluye referencia a su ubicación original.

### Paso 9 — Constantes y Diccionarios

**Ubicación en modelo_sentiment.ipynb:** PARTE 2, Celdas 19-21

Estas constantes deben incluirse en `src/preprocessing.py`.

In [ ]:
# ============================================================================
# CONSTANTES Y DICCIONARIOS (del modelo_sentiment.ipynb PARTE 2)
# ============================================================================
# Ubicación: modelo_sentiment.ipynb, Celdas 19-21 (Líneas ~600-850)
# Estas constantes deben incluirse en src/preprocessing.py
# ============================================================================

import re

# ──────────────────────────────────────────────────────────────────────
# PATRÓN REGEX PARA EMOJIS
# ──────────────────────────────────────────────────────────────────────
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F700-\U0001F77F"  # alchemical symbols
    "\U0001F780-\U0001F7FF"  # geometric shapes extended
    "\U0001F800-\U0001F8FF"  # supplemental arrows-C
    "\U0001F900-\U0001F9FF"  # supplemental symbols and pictographs
    "\U0001FA00-\U0001FA6F"  # chess symbols
    "\U0001FA70-\U0001FAFF"  # symbols and pictographs extended-A
    "\U00002702-\U000027B0"  # dingbats
    "\U00002300-\U000023FF"  # miscellaneous technical
    "\U0001F1E0-\U0001F1FF"  # flags (iOS)
    "]+",
    flags=re.UNICODE
)

# ──────────────────────────────────────────────────────────────────────
# DICCIONARIO: NÚMEROS A TEXTO (1-100)
# ──────────────────────────────────────────────────────────────────────
NUMEROS_TEXTO = {
    '0': 'cero', '1': 'uno', '2': 'dos', '3': 'tres', '4': 'cuatro',
    '5': 'cinco', '6': 'seis', '7': 'siete', '8': 'ocho', '9': 'nueve',
    '10': 'diez', '11': 'once', '12': 'doce', '13': 'trece', '14': 'catorce',
    '15': 'quince', '16': 'dieciséis', '17': 'diecisiete', '18': 'dieciocho',
    '19': 'diecinueve', '20': 'veinte', '21': 'veintiuno', '22': 'veintidós',
    '23': 'veintitrés', '24': 'veinticuatro', '25': 'veinticinco',
    '30': 'treinta', '40': 'cuarenta', '50': 'cincuenta', '60': 'sesenta',
    '70': 'setenta', '80': 'ochenta', '90': 'noventa', '100': 'cien'
}

# ──────────────────────────────────────────────────────────────────────
# DICCIONARIO: ORDINALES (1-10)
# ──────────────────────────────────────────────────────────────────────
ORDINALES = {
    '1': 'primer', '2': 'segundo', '3': 'tercer', '4': 'cuarto', '5': 'quinto',
    '6': 'sexto', '7': 'séptimo', '8': 'octavo', '9': 'noveno', '10': 'décimo'
}

# ──────────────────────────────────────────────────────────────────────
# TOKEN PATTERN PARA TF-IDF
# ──────────────────────────────────────────────────────────────────────
# Captura palabras normales Y tokens especiales <TOKEN>
TOKEN_PATTERN = r'<[A-Z_0-9]+>|\b\w\w+\b'

print("✅ Constantes definidas:")
print(f"   - EMOJI_PATTERN: Regex compilado")
print(f"   - NUMEROS_TEXTO: {len(NUMEROS_TEXTO)} números")
print(f"   - ORDINALES: {len(ORDINALES)} ordinales")
print(f"   - TOKEN_PATTERN: '{TOKEN_PATTERN}'")

### Paso 10 — Las 7 Funciones de Preprocesamiento

**Ubicación en modelo_sentiment.ipynb:** PARTE 2, Celdas 19-33

Estas son las funciones EXACTAS que se usan en el modelo. Copiar directamente a `src/preprocessing.py`.

In [ ]:
# ============================================================================
# LAS 7 FUNCIONES DE PREPROCESAMIENTO (del modelo_sentiment.ipynb PARTE 2)
# ============================================================================
# Ubicación: modelo_sentiment.ipynb, Celdas 19-33 (Líneas ~850-1850)
# Estas funciones deben incluirse en src/preprocessing.py
# Orden de ejecución: 1→2→3→4→5→6→7
# ============================================================================

import pandas as pd

def numero_a_texto(num_str):
    """Convierte un número string a texto español (función auxiliar)."""
    return NUMEROS_TEXTO.get(num_str, num_str)

# ──────────────────────────────────────────────────────────────────────
# FUNCIÓN 1: limpiar_ruido()
# Celda 19 de modelo_sentiment.ipynb (línea ~856)
# ──────────────────────────────────────────────────────────────────────
def limpiar_ruido(texto):
    """
    Elimina elementos de ruido del texto que no aportan valor semántico.
    
    IMPORTANTE: Este paso se ejecuta PRIMERO para eliminar caracteres < > 
    ANTES de la tokenización, evitando conflictos con tokens especiales.
    
    Elimina:
        - URLs (http, https, www)
        - Correos electrónicos
        - Menciones (@usuario)
        - Hashtags (#tema)
        - Caracteres < y > del texto original
    """
    if pd.isna(texto):
        return texto
    
    texto = str(texto)
    
    # Eliminar URLs
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
    
    # Eliminar correos electrónicos
    texto = re.sub(r'\S+@\S+\.\S+', '', texto)
    
    # Eliminar menciones (@usuario)
    texto = re.sub(r'@\w+', '', texto)
    
    # Eliminar hashtags (#tema)
    texto = re.sub(r'#\w+', '', texto)
    
    # Eliminar caracteres < y > (CRÍTICO: antes de tokenización)
    texto = re.sub(r'[<>]', '', texto)
    
    # Normalizar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# ──────────────────────────────────────────────────────────────────────
# FUNCIÓN 2: eliminar_emojis()
# Celda 21 de modelo_sentiment.ipynb (línea ~988)
# ──────────────────────────────────────────────────────────────────────
def eliminar_emojis(texto):
    """
    Elimina todos los emojis del texto.
    Decisión: Eliminar (frecuencia 0.38%, Cramér's V = 0.0078)
    """
    if pd.isna(texto):
        return texto
    
    texto = EMOJI_PATTERN.sub('', str(texto))
    # Normalizar espacios
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# ──────────────────────────────────────────────────────────────────────
# FUNCIÓN 3: tokenizar_caracteres_especiales()
# Celda 23 de modelo_sentiment.ipynb (línea ~1083)
# ──────────────────────────────────────────────────────────────────────
def tokenizar_caracteres_especiales(texto):
    """
    Convierte patrones de precio y porcentaje a tokens especiales.
    
    Transformaciones:
        - número + € → <PRECIO>  (15.99€, 20 €)
        - € + número → <PRECIO>  (€15.99)
        - $ + número → <PRECIO>  ($29.99)
        - número + $ → <PRECIO>  (29.99$)
        - número + % → <PORCENTAJE>  (50%, 100 %)
    """
    if pd.isna(texto):
        return texto
    
    texto = str(texto)
    
    # Precio: número + € (con o sin espacio)
    texto = re.sub(r'\d+[.,]?\d*\s*€', ' <PRECIO> ', texto)
    
    # Precio: € + número
    texto = re.sub(r'€\s*\d+[.,]?\d*', ' <PRECIO> ', texto)
    
    # Precio: $ + número
    texto = re.sub(r'\$\s*\d+[.,]?\d*', ' <PRECIO> ', texto)
    
    # Precio: número + $
    texto = re.sub(r'\d+[.,]?\d*\s*\$', ' <PRECIO> ', texto)
    
    # Porcentaje: número + % (con o sin espacio)
    texto = re.sub(r'\d+\s*%', ' <PORCENTAJE> ', texto)
    
    # Normalizar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# ──────────────────────────────────────────────────────────────────────
# FUNCIÓN 4: tokenizar_patrones_numericos()
# Celda 25 de modelo_sentiment.ipynb (línea ~1240)
# ──────────────────────────────────────────────────────────────────────
def tokenizar_patrones_numericos(texto):
    """
    Transforma patrones numéricos con significado semántico a texto.
    Elimina números aislados sin contexto.
    
    Orden interno (CRÍTICO):
        1. Transformar patrones con contexto (duración, cantidad, etc.)
        2. Preservar patrones especiales (24h, cero problemas)
        3. Eliminar números aislados restantes
    """
    if pd.isna(texto):
        return texto
    
    texto = str(texto)
    
    # Duración: "2 días" → "dos días"
    def reemplazar_duracion(match):
        num = match.group(1)
        unidad = match.group(2).lower()
        texto_num = numero_a_texto(num)
        return f"{texto_num} {unidad}"
    
    texto = re.sub(
        r'\b(\d{1,2})\s*(días?|semanas?|meses?|años?|horas?|minutos?)\b',
        reemplazar_duracion, texto, flags=re.IGNORECASE
    )
    
    # Número de veces: "3 veces" → "tres veces"
    def reemplazar_veces(match):
        num = match.group(1)
        texto_num = numero_a_texto(num)
        return f"{texto_num} veces"
    
    texto = re.sub(r'\b(\d{1,2})\s*veces\b', reemplazar_veces, texto, flags=re.IGNORECASE)
    
    # Cantidad: "5 unidades" → "cinco unidades"
    def reemplazar_cantidad(match):
        num = match.group(1)
        unidad = match.group(2).lower()
        texto_num = numero_a_texto(num)
        return f"{texto_num} {unidad}"
    
    texto = re.sub(
        r'\b(\d{1,2})\s*(unidades?|piezas?|productos?|artículos?|paquetes?)\b',
        reemplazar_cantidad, texto, flags=re.IGNORECASE
    )
    
    # Ordinales typo: "3er" → "tercer"
    def reemplazar_ordinal(match):
        num = match.group(1)
        return ORDINALES.get(num, match.group(0))
    
    texto = re.sub(r'\b(\d)(er|do|ro|to|vo|mo|no)\b', reemplazar_ordinal, texto, flags=re.IGNORECASE)
    
    # Estrellas: "5 estrellas" → "cinco estrellas"
    def reemplazar_estrellas(match):
        num = match.group(1)
        texto_num = numero_a_texto(num)
        return f"{texto_num} estrellas"
    
    texto = re.sub(r'\b([1-5])\s*estrellas?\b', reemplazar_estrellas, texto, flags=re.IGNORECASE)
    
    # Eliminar números aislados (sin contexto semántico)
    texto = re.sub(r'\b\d+[.,]?\d*\b', '', texto)
    
    # Normalizar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# ──────────────────────────────────────────────────────────────────────
# FUNCIÓN 5: tokenizar_puntuacion_repetida()
# Celda 27 de modelo_sentiment.ipynb (línea ~1447)
# ──────────────────────────────────────────────────────────────────────
def tokenizar_puntuacion_repetida(texto):
    """
    Convierte puntuación repetida a tokens que capturan intensidad emocional.
    
    IMPORTANTE: El orden de las regex es crítico (más largo primero).
    """
    if pd.isna(texto):
        return texto
    
    texto = str(texto)
    
    # Exclamaciones (orden: más largo primero)
    texto = re.sub(r'!{4,}', ' <EXCL_MULT> ', texto)   # 4 o más → extremo
    texto = re.sub(r'!{3}', ' <EXCL_3> ', texto)       # exactamente 3 → alto
    texto = re.sub(r'!{2}', ' <EXCL_2> ', texto)       # exactamente 2 → moderado
    
    # Interrogaciones (orden: más largo primero)
    texto = re.sub(r'\?{4,}', ' <INTER_MULT> ', texto)  # 4 o más → extremo
    texto = re.sub(r'\?{3}', ' <INTER_3> ', texto)      # exactamente 3 → alto
    texto = re.sub(r'\?{2}', ' <INTER_2> ', texto)      # exactamente 2 → moderado
    
    # Puntos suspensivos (orden: más largo primero)
    texto = re.sub(r'\.{4,}', ' <SUSP_MULT> ', texto)  # 4 o más puntos
    texto = re.sub(r'\.{3}', ' <SUSP_3> ', texto)      # exactamente 3 puntos
    
    # Puntuación mixta (!?, ?!, ¡¿, etc.)
    texto = re.sub(r'[!?¡¿]{2,}', ' <MIXTO> ', texto)
    
    # Normalizar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# ──────────────────────────────────────────────────────────────────────
# FUNCIÓN 6: eliminar_caracteres_restantes()
# Celda 29 de modelo_sentiment.ipynb (línea ~1570)
# ──────────────────────────────────────────────────────────────────────
def eliminar_caracteres_restantes(texto):
    """
    Elimina caracteres especiales que no aportan valor semántico.
    
    Caracteres a eliminar: ( ) - / + * = # & ^ ~ \ | [ ] { } @
    
    NOTA: 
    - NO eliminar < > porque los tokens especiales <TOKEN> deben preservarse
    - NO eliminar _ dentro de tokens especiales
    """
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    
    # Lista de tokens especiales a preservar
    tokens_especiales = [
        '<PRECIO>', '<PORCENTAJE>', 
        '<EXCL_2>', '<EXCL_3>', '<EXCL_MULT>',
        '<INTER_2>', '<INTER_3>', '<INTER_MULT>',
        '<SUSP_3>', '<SUSP_MULT>', '<MIXTO>'
    ]
    
    # Reemplazar tokens especiales con placeholders únicos
    placeholders = {}
    for i, token in enumerate(tokens_especiales):
        placeholder = f"PLACEHOLDER{i}HOLDER"
        placeholders[placeholder] = token
        texto = texto.replace(token, placeholder)
    
    # Caracteres a eliminar (sin < >)
    # Incluye: ( ) - / + * = # & ^ ~ \ | [ ] { } @ _ £ ¥
    caracteres_eliminar = r'[\(\)\-/\+\*=#&\^~\\|\[\]\{\}@£¥_]'
    texto = re.sub(caracteres_eliminar, ' ', texto)
    
    # Restaurar tokens especiales
    for placeholder, token in placeholders.items():
        texto = texto.replace(placeholder, token)
    
    # Normalizar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

# ──────────────────────────────────────────────────────────────────────
# FUNCIÓN 7: normalizar_mayusculas()
# Celda 31 de modelo_sentiment.ipynb (línea ~1683)
# ──────────────────────────────────────────────────────────────────────
def normalizar_mayusculas(texto):
    """
    Normaliza mayúsculas preservando palabras ALL-CAPS.
    
    Lógica:
    - Palabras completamente en mayúsculas (≥2 chars): PRESERVAR
    - Resto: convertir a minúsculas
    
    Razón: ALL-CAPS indica énfasis emocional importante para sentiment
    Ejemplos: "EXCELENTE", "HORRIBLE", "NO"
    """
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    palabras = texto.split()
    resultado = []
    
    for palabra in palabras:
        # Preservar tokens especiales como están
        if palabra.startswith('<') and palabra.endswith('>'):
            resultado.append(palabra)
        # Preservar palabras ALL-CAPS de 2+ caracteres
        elif palabra.isupper() and len(palabra) >= 2:
            resultado.append(palabra)
        else:
            resultado.append(palabra.lower())
    
    return ' '.join(resultado)

# ──────────────────────────────────────────────────────────────────────
# FUNCIÓN MAESTRA: preprocesar_texto()
# Celda 33 de modelo_sentiment.ipynb (línea ~1791)
# ──────────────────────────────────────────────────────────────────────
def preprocesar_texto(texto):
    """
    Pipeline completo de preprocesamiento para análisis de sentimiento.
    
    Orden de ejecución (según PLAN_MODELADO.md):
    1. limpiar_ruido() - URLs, emails, menciones, hashtags, <>
    2. eliminar_emojis() - Todos los emojis
    3. tokenizar_caracteres_especiales() - € $ % → tokens
    4. tokenizar_patrones_numericos() - Números → texto/eliminación
    5. tokenizar_puntuacion_repetida() - !!! ??? ... → tokens
    6. eliminar_caracteres_restantes() - Caracteres sin valor
    7. normalizar_mayusculas() - Preservar ALL-CAPS
    
    Returns:
        String preprocesado listo para vectorización
    """
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    
    # Pipeline en orden
    texto = limpiar_ruido(texto)
    texto = eliminar_emojis(texto)
    texto = tokenizar_caracteres_especiales(texto)
    texto = tokenizar_patrones_numericos(texto)
    texto = tokenizar_puntuacion_repetida(texto)
    texto = eliminar_caracteres_restantes(texto)
    texto = normalizar_mayusculas(texto)
    
    return texto

print("\n✅ Las 7 funciones de preprocesamiento definidas:")
print("   1. limpiar_ruido()")
print("   2. eliminar_emojis()")
print("   3. tokenizar_caracteres_especiales()")
print("   4. tokenizar_patrones_numericos()")
print("   5. tokenizar_puntuacion_repetida()")
print("   6. eliminar_caracteres_restantes()")
print("   7. normalizar_mayusculas()")
print("   + preprocesar_texto() (función maestra)")

### Paso 11 — Test del Pipeline Completo

Prueba del pipeline completo con ejemplos representativos.

In [ ]:
# ============================================================================
# TEST DEL PIPELINE COMPLETO
# ============================================================================
# Verificación de que todas las funciones trabajan correctamente juntas
# ============================================================================

print("=" * 80)
print("TEST COMPLETO DEL PIPELINE DE PREPROCESAMIENTO")
print("=" * 80)

textos_test = [
    "EXCELENTE!!! producto por 25€, 100% recomendado https://amazon.es @user",
    "Muy malo... no vale los 50$ que cuesta 😡😡😡 #estafa",
    "Duración de 3 horas, calidad 5 estrellas ***** MUY BUENO",
    "Regular, ni bueno ni malo. Precio/calidad aceptable",
    "NO LO COMPRES!!!! Es una BASURA total (devuelto)",
]

for i, texto in enumerate(textos_test, 1):
    resultado = preprocesar_texto(texto)
    print(f"\n{'─'*80}")
    print(f"EJEMPLO {i}")
    print(f"{'─'*80}")
    print(f"ORIGINAL:  '{texto}'")
    print(f"PROCESADO: '{resultado}'")

print(f"\n{'='*80}")
print("✅ PIPELINE DE PREPROCESAMIENTO VERIFICADO")
print("="*80)

## PARTE E: GENERACIÓN DE ARTEFACTOS DEL MODELO

⚠️ **NOTA IMPORTANTE**: Los artefactos (`modelo_sentiment_final.joblib`, `tfidf_vectorizer_final.joblib`) ya existen y fueron generados siguiendo el flujo completo de [modelo_sentiment.ipynb](modelo_sentiment.ipynb):

### Flujo de generación (ejecutado en modelo_sentiment.ipynb):

1. **PARTE 1**: Carga Amazon Reviews Multi (205K train)
2. **PARTE 2**: Preprocesamiento (7 funciones)
3. **PARTE 3-7**: Experimentación y optimización de hiperparámetros
4. **PARTE 8**: Limpieza de datos atípicos (elimina 727 muestras con ruido de etiquetado, P≥0.9)
5. **PARTE 9**: Entrenamiento final con datos limpios y guardado de artefactos

### Artefactos generados:
- ✅ `stopwords_eliminar.txt` (403 palabras)
- ✅ `modelo_sentiment_final.joblib` (LogisticRegression C=0.5, balanced, entrenado en ~204K muestras limpias)
- ✅ `tfidf_vectorizer_final.joblib` (TfidfVectorizer 10K features, (1,2)-gramas, 403 stopwords)

**Si necesitas regenerar los artefactos**, ejecuta las PARTES 1-9 del notebook [modelo_sentiment.ipynb](modelo_sentiment.ipynb).

### Verificación de artefactos

Verifica que los archivos existen y cárgalos para inspección:

In [5]:
import os
import joblib

print("🔍 VERIFICACIÓN DE ARTEFACTOS")
print("=" * 80)

# Lista de archivos requeridos
archivos_requeridos = [
    'stopwords_eliminar.txt',
    'modelo_sentiment_final.joblib',
    'tfidf_vectorizer_final.joblib'
]

# Verificar existencia
for archivo in archivos_requeridos:
    existe = os.path.exists(archivo)
    simbolo = "✅" if existe else "❌"
    tamanio = f"({os.path.getsize(archivo) / 1024:.1f} KB)" if existe else ""
    print(f"{simbolo} {archivo} {tamanio}")

# Si existen los modelos, cargarlos y mostrar info
if all(os.path.exists(f) for f in archivos_requeridos):
    print("\n📦 CARGANDO MODELOS...")
    
    # Cargar vectorizador
    vectorizer = joblib.load('tfidf_vectorizer_final.joblib')
    print(f"\n   TfidfVectorizer:")
    print(f"      Vocabulario: {len(vectorizer.vocabulary_):,} términos")
    print(f"      max_features: {vectorizer.max_features}")
    print(f"      ngram_range: {vectorizer.ngram_range}")
    print(f"      Stopwords: {len(vectorizer.stop_words_) if vectorizer.stop_words_ else 0}")
    
    # Cargar modelo
    model = joblib.load('modelo_sentiment_final.joblib')
    print(f"\n   LogisticRegression:")
    print(f"      C: {model.C}")
    print(f"      Clases: {model.classes_}")
    print(f"      Features: {model.coef_.shape}")
    
    print("\n✅ Todos los artefactos están listos para producción!")
else:
    print("\n⚠️  ARTEFACTOS FALTANTES")
    print("   Ejecuta las PARTES 1-9 de modelo_sentiment.ipynb para generarlos.")

🔍 VERIFICACIÓN DE ARTEFACTOS
✅ stopwords_eliminar.txt (4.2 KB)
❌ modelo_sentiment_final.joblib 
❌ tfidf_vectorizer_final.joblib 

⚠️  ARTEFACTOS FALTANTES
   Ejecuta las PARTES 1-9 de modelo_sentiment.ipynb para generarlos.


### Script de generación de artefactos

Si los archivos `.joblib` no existen, este script los genera usando el flujo exacto de [modelo_sentiment.ipynb](modelo_sentiment.ipynb):

**⏱️ Tiempo estimado**: ~15-20 minutos (carga 205K muestras, preprocesa, entrena)

In [6]:
# ============================================================================
# GENERACIÓN DE ARTEFACTOS - SCRIPT COMPLETO
# ============================================================================
# Este código replica el flujo de modelo_sentiment.ipynb PARTES 1-9

import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib
import time

print("=" * 80)
print("🚀 INICIO: GENERACIÓN DE ARTEFACTOS DEL MODELO")
print("=" * 80)

# ────────────────────────────────────────────────────────────────────────────
# PASO 1: CARGAR DATOS (PARTE 1 de modelo_sentiment.ipynb)
# ────────────────────────────────────────────────────────────────────────────
print("\n📥 PASO 1: Cargando Amazon Reviews Multi (Spanish)...")
start_total = time.time()

dataset = load_dataset("amazon_reviews_multi", "es", trust_remote_code=True)

# Convertir a DataFrame
df_train_raw = pd.DataFrame(dataset['train'])      # 200K
df_validation_raw = pd.DataFrame(dataset['validation'])  # 5K
df_test_raw = pd.DataFrame(dataset['test'])        # 5K

# Combinar train + validation → df_dev (205K)
df_dev = pd.concat([df_train_raw, df_validation_raw], ignore_index=True)

print(f"   ✅ df_dev: {len(df_dev):,} muestras")
print(f"   ✅ df_test: {len(df_test_raw):,} muestras")

# Crear variable target 'sentiment'
def crear_sentimiento(stars):
    if stars in [1, 2]:
        return 'negativo'
    elif stars == 3:
        return 'neutro'
    else:  # 4, 5
        return 'positivo'

df_dev['sentiment'] = df_dev['stars'].apply(crear_sentimiento)
df_test_raw['sentiment'] = df_test_raw['stars'].apply(crear_sentimiento)

print(f"\n   Distribución en df_dev:")
print(df_dev['sentiment'].value_counts())

# ────────────────────────────────────────────────────────────────────────────
# PASO 2: PREPROCESAR (PARTE 2 de modelo_sentiment.ipynb)
# ────────────────────────────────────────────────────────────────────────────
print("\n🔧 PASO 2: Preprocesando textos (7 funciones)...")
start_prep = time.time()

# Aplicar preprocesar_texto() definido en PARTE C arriba
df_dev['texto_limpio'] = df_dev['review_body'].apply(preprocesar_texto)
df_test_raw['texto_limpio'] = df_test_raw['review_body'].apply(preprocesar_texto)

print(f"   ✅ Preprocesamiento completo ({time.time() - start_prep:.1f}s)")

# ────────────────────────────────────────────────────────────────────────────
# PASO 3: SPLIT TRAIN/HOLDOUT (PARTE 1 de modelo_sentiment.ipynb)
# ────────────────────────────────────────────────────────────────────────────
print("\n📊 PASO 3: Creando splits estratificados (85/15)...")

X_stack_train, X_holdout, y_stack_train, y_holdout = train_test_split(
    df_dev['texto_limpio'],
    df_dev['sentiment'],
    test_size=0.15,
    stratify=df_dev['sentiment'],
    random_state=42
)

print(f"   ✅ X_stack_train: {len(X_stack_train):,} muestras")
print(f"   ✅ X_holdout: {len(X_holdout):,} muestras")

# ────────────────────────────────────────────────────────────────────────────
# PASO 4: VECTORIZACIÓN (PARTE 9 de modelo_sentiment.ipynb)
# ────────────────────────────────────────────────────────────────────────────
print("\n📊 PASO 4: Creando TF-IDF Vectorizer...")

tfidf_vectorizer_final = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    token_pattern=TOKEN_PATTERN,  # Definido en PARTE C
    min_df=5,
    max_df=0.95,
    stop_words=list(STOPWORDS_ELIMINAR),  # 403 palabras
    sublinear_tf=True
)

start_vec = time.time()
X_train_tfidf = tfidf_vectorizer_final.fit_transform(X_stack_train)
X_holdout_tfidf = tfidf_vectorizer_final.transform(X_holdout)

print(f"   ✅ Vectorización completa ({time.time() - start_vec:.1f}s)")
print(f"   ✅ Vocabulario: {len(tfidf_vectorizer_final.vocabulary_):,} términos")
print(f"   ✅ Shape: {X_train_tfidf.shape}")

# ────────────────────────────────────────────────────────────────────────────
# PASO 5: ENTRENAR MODELO (PARTE 9 de modelo_sentiment.ipynb)
# ────────────────────────────────────────────────────────────────────────────
print("\n🤖 PASO 5: Entrenando LogisticRegression...")

modelo_sentiment_final = LogisticRegression(
    C=0.5,
    solver='lbfgs',
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

start_train = time.time()
modelo_sentiment_final.fit(X_train_tfidf, y_stack_train)
print(f"   ✅ Entrenamiento completo ({time.time() - start_train:.1f}s)")

# Evaluar en holdout
from sklearn.metrics import classification_report
y_pred_holdout = modelo_sentiment_final.predict(X_holdout_tfidf)
print("\n   📊 Evaluación en Holdout:")
print(classification_report(y_holdout, y_pred_holdout, digits=4))

# ────────────────────────────────────────────────────────────────────────────
# PASO 6: GUARDAR ARTEFACTOS
# ────────────────────────────────────────────────────────────────────────────
print("\n💾 PASO 6: Guardando artefactos...")

joblib.dump(modelo_sentiment_final, 'modelo_sentiment_final.joblib')
joblib.dump(tfidf_vectorizer_final, 'tfidf_vectorizer_final.joblib')

print(f"   ✅ modelo_sentiment_final.joblib")
print(f"   ✅ tfidf_vectorizer_final.joblib")
print(f"   ✅ stopwords_eliminar.txt (ya existe)")

print("\n" + "=" * 80)
print(f"✅ GENERACIÓN COMPLETA - Tiempo total: {time.time() - start_total:.1f}s")
print("=" * 80)

ModuleNotFoundError: No module named 'datasets'

---

## Parte D · Mapa de Referencias Completo

Esta sección documenta EXACTAMENTE dónde se encuentran todos los componentes del modelo en `modelo_sentiment.ipynb`.

### Mapa Completo de Referencias a modelo_sentiment.ipynb

| Componente | Ubicación en modelo_sentiment.ipynb | Celda # | Líneas Aprox |
|------------|-------------------------------------|---------|--------------|
| **CONSTANTES** |
| `EMOJI_PATTERN` | PARTE 2, Celda 21 | 21 | ~610-630 |
| `NUMEROS_TEXTO` | PARTE 2, Celda 25 | 25 | ~760-780 |
| `ORDINALES` | PARTE 2, Celda 25 | 25 | ~780-790 |
| `TOKEN_PATTERN` | PARTE 3, Celda 44 | 44 | ~1433 |
| **FUNCIONES DE PREPROCESAMIENTO** |
| `limpiar_ruido()` | PARTE 2, Celda 19 | 19 | ~856-890 |
| `eliminar_emojis()` | PARTE 2, Celda 21 | 21 | ~988-1010 |
| `tokenizar_caracteres_especiales()` | PARTE 2, Celda 23 | 23 | ~1083-1120 |
| `tokenizar_patrones_numericos()` | PARTE 2, Celda 25 | 25 | ~1240-1380 |
| `tokenizar_puntuacion_repetida()` | PARTE 2, Celda 27 | 27 | ~1447-1490 |
| `eliminar_caracteres_restantes()` | PARTE 2, Celda 29 | 29 | ~1570-1620 |
| `normalizar_mayusculas()` | PARTE 2, Celda 31 | 31 | ~1683-1720 |
| `preprocesar_texto()` | PARTE 2, Celda 33 | 33 | ~1791-1810 |
| **ARTEFACTOS** |
| `modelo_sentiment_final.joblib` | PARTE 9, Celda 173 | 173 | ~6898-6920 |
| `tfidf_vectorizer_final.joblib` | PARTE 9, Celda 173 | 173 | ~6898-6920 |
| Guardado de artefactos | PARTE 9 | 173 | ~6898-6920 |
| **CONFIGURACIÓN TF-IDF** |
| `max_features=10000` | PARTE 4, Celda 92 | 92 | ~2727-2740 |
| `ngram_range=(1,2)` | PARTE 4, Celda 93 | 93 | ~2740-2770 |
| Token pattern configuración | PARTE 3, Celda 44 | 44 | ~1433-1450 |
| **CONFIGURACIÓN MODELO** |
| `LogisticRegression(C=0.5)` | PARTE 4, Celda 95 | 95 | ~2807-2840 |
| `class_weight='balanced'` | PARTE 4, Celda 95 | 95 | ~2807-2840 |
| **EVALUACIÓN** |
| Evaluación en TEST | PARTE 9, Celdas 166-170 | 166-170 | ~6478-6650 |
| Reseñas controladas | PARTE 7, Celdas 63-66 | 63-66 | ~1969-2040 |
| Conclusiones finales | PARTE 11, Celda 192 | 192 | ~8470-8510 |

**NOTA CRÍTICA:** Las funciones de preprocesamiento están en la **PARTE 2** (no en PARTE 11). La PARTE 11 solo contiene la especificación técnica y referencias.

---

## Resumen Ejecutivo del Plan de Producción

### ✅ Componentes Listos para Implementación

Este notebook contiene TODO lo necesario para pasar el modelo a producción:

#### 📦 **Parte A: Checklist de Implementación** (Celdas 3-4)
- 8 pasos detallados con validaciones/riesgos
- Artefactos necesarios identificados
- Estructura de archivos definida

#### 🔧 **Parte B: Código de Módulos** (Celdas 5-20)
- `src/preprocessing.py` - ACTUALIZADO con funciones placeholder
- `src/config.py` - Configuración centralizada
- `src/model_loader.py` - Carga de artefactos
- `src/keyword_extractor.py` - Extracción de keywords
- `src/predictor.py` - Clase principal de inferencia
- `src/api_schema.py` - Contrato API
- `src/logging_config.py` - Logging estructurado
- `tests/test_integration.py` - Tests de integración

#### 🎯 **Parte C: Funciones REALES del Modelo** (Celdas 21-24)
- ✅ Constantes: EMOJI_PATTERN, NUMEROS_TEXTO, ORDINALES, TOKEN_PATTERN
- ✅ Las 7 funciones de preprocesamiento EXACTAS de modelo_sentiment.ipynb
- ✅ Función maestra `preprocesar_texto()`
- ✅ Test del pipeline completo

#### 📍 **Parte D: Mapa de Referencias** (Celda 25)
- Tabla completa con ubicación exacta de cada componente
- Referencias cruzadas a modelo_sentiment.ipynb
- Números de celda y líneas aproximadas

---

### 🚀 Próximos Pasos

1. **Actualizar `src/preprocessing.py`**
   - Reemplazar código placeholder con funciones de Parte C (Celda 23)
   - Copiar constantes de Celda 22

2. **Copiar artefactos**
   - `modelo_sentiment_final.joblib` → raíz del proyecto
   - `tfidf_vectorizer_final.joblib` → raíz del proyecto
   - `stopwords_eliminar.txt` → raíz del proyecto (403 palabras)

3. **Ejecutar tests**
   ```bash
   pytest tests/test_integration.py -v
   ```

4. **Crear API (FastAPI/Flask)**
   - Importar `SentimentPredictor` de `src/predictor.py`
   - Endpoint POST `/predict` con contrato de `src/api_schema.py`

---

### 📖 Documentación de Referencias

- **PLAN_MODELADO.md**: Especificaciones técnicas y arquitectura
- **modelo_sentiment.ipynb PARTE 2**: Funciones de preprocesamiento (origen)
- **modelo_sentiment.ipynb PARTE 9**: Guardado de artefactos
- **modelo_sentiment.ipynb PARTE 11**: Especificación para producción

---

**Nota Importante:** Las funciones de preprocesamiento en la Parte C (Celda 23) son las **EXACTAS** que se usaron para entrenar el modelo. Cualquier modificación afectará las predicciones.